# EmotionLab — Construcción del Dataset Híbrido

## Proyecto Capstone — Ingeniería en Informática

**Objetivo:** desarrollar un dataset híbrido a partir de datos reales obtenidos mediante las evaluaciones de EmotionLab, destinado al entrenamiento de un modelo de Machine Learning que permita estimar la intención declarada de reutilización del programa.

**Metodología:** extracción, análisis exploratorio, limpieza, normalización, integración relacional, seudonimización y validación de datos.

**Tecnologías:** Python, Pandas, Google Colab y Scikit-learn.

**Principio metodológico:** conservar la procedencia y los valores originales, documentando las transformaciones y los criterios utilizados para tratar registros incompletos.

# **Imports**

In [ ]:
import pandas as pd
from google.colab import files
import zipfile
import numpy as np

# Carga de Archivo

In [ ]:
# Cargar el archivo original
uploaded = files.upload()

Saving Datos rescatados.xlsx to Datos rescatados (1).xlsx


In [ ]:
# Leer las hojas disponibles
archivo = pd.ExcelFile("Datos rescatados.xlsx")

print("Hojas disponibles:")
print(archivo.sheet_names)

# Examinar la estructura de cada hoja
for hoja in archivo.sheet_names:
    df = pd.read_excel(
        "Datos rescatados.xlsx",
        sheet_name=hoja,
        header=None
    )

    print(f"\nHoja: {hoja}")
    print("Dimensiones:", df.shape)
    display(df.head(10))


Hojas disponibles:
['Hoja 1']

Hoja: Hoja 1
Dimensiones: (39, 26)


,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,Cursos,Fecha,Nombre,Edad,Hora,NaN,Formulario,Luciano Ayala,Armando Aguirre,Sebastian Vasquez,...,session_bc95768bae7f2daae2eebbe9,session_4d67a791bbdefc046cdf867f,session_61708cabed43caad09f81b70,session_c379e1536f33eb5fab60f187,session_3b43e5a31bc5afd36495f50c,session_e022a6c404f2ccfe24180f11,session_78f530d537926359071038a3,session_dfc1f14276b88caf8666efaf,session_6c179db758d0763d459f4a21,session_a4a4f71b372c9c7c65eacaf2
1,3ro C,2026-10-29 00:00:00,Luciano Ayala,17,11:47:00,NaN,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2,3ro C,2026-10-29 00:00:00,Armando Aguirre,17,11:50:00,NaN,edad,17,17,17,...,16,16,17,NaN,NaN,17,16,17,17,17
3,3ro C,2026-11-29 00:00:00,Sebastian Vasquez,17,12:05:00,NaN,genero,m,m,m,...,f,f,NaN,NaN,NaN,m,f,NaN,NaN,f
4,3ro C,2026-11-29 00:00:00,Mia zuaznabar,17,12:05:00,NaN,2026-09-01 00:00:00,1,2,1,...,8,9,NaN,NaN,NaN,9,8,4,NaN,5
5,3ro C,2026-11-29 00:00:00,Sara Cerda,16,NaN,NaN,2026-09-01 00:00:00,3,3,1,...,8,9,NaN,NaN,NaN,6,7,NaN,NaN,2
6,NaN,NaN,NaN,NaN,NaN,NaN,2026-09-01 00:00:00,4,1,9,...,8,1,NaN,NaN,NaN,6,5,NaN,NaN,4
7,NaN,NaN,NaN,NaN,NaN,NaN,2026-09-01 00:00:00,3,4,1,...,9,9,NaN,NaN,NaN,9,7,NaN,NaN,3
8,4to A,2026-10-29 00:00:00,Karoll Rotter,18,NaN,NaN,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
9,4to A,2026-10-29 00:00:00,Jhissel Aravena,18,NaN,NaN,2026-09-01 00:00:00,1,2,1,...,8,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Etapa 1: Normalización y seudonimización de participantes

**Objetivo:** transformar los registros originales de EmotionLab en una estructura tabular uniforme, adecuada para su posterior procesamiento y entrenamiento de modelos de Machine Learning.

**Procedimiento:**

Se identifican los bloques de formularios y se extraen las respuestas asociadas a cada participante, manteniendo la correspondencia entre estudiante, formulario y pregunta.

Posteriormente, se asignan identificadores seudonimizados a los participantes y se construye una tabla estructurada donde cada fila representa a un estudiante.

**Tratamiento de datos incompletos:**

Los registros se clasifican según su nivel de completitud:

- Completo: contiene las 12 respuestas.
- Parcial: contiene entre 1 y 11 respuestas.
- Sin respuestas: no contiene respuestas registradas.

No se eliminan ni imputan valores en esta etapa, con el propósito de preservar la integridad de los datos reales y permitir una evaluación posterior de su calidad.

In [ ]:

# Cargar el archivo original sin modificarlo
base = pd.read_excel(
    "Datos rescatados.xlsx",
    sheet_name=0,
    header=None
)

# Localizar el inicio de cada bloque de formularios
inicios = base.index[
    base.iloc[:, 6].astype(str)
    .str.strip().str.lower().eq("formulario")
].tolist()

limites = inicios[1:] + [len(base)]

participantes = []
respuestas = []
mapa_privado = []

for bloque, (inicio, fin) in enumerate(
    zip(inicios, limites), start=1
):
    # Localizar los formularios 1, 2 y 3
    etapas = {}

    for fila in range(inicio + 1, fin):
        etiqueta = str(base.iat[fila, 6]).strip()

        if etiqueta in ("1", "2", "3", "1.0", "2.0", "3.0"):
            etapas[int(float(etiqueta))] = fila

    assert set(etapas) == {1, 2, 3}, (
        f"Formularios incompletos en bloque {bloque}"
    )

    # Identificar las cuatro preguntas de cada formulario
    preguntas = {}

    for formulario in (1, 2, 3):
        inicio_f = etapas[formulario]
        fin_f = etapas.get(formulario + 1, fin)

        filas = [
            r for r in range(inicio_f + 1, fin_f)
            if pd.notna(base.iat[r, 6])
            and str(base.iat[r, 6]).strip().lower()
            not in ("edad", "genero")
        ]

        assert len(filas) == 4, (
            f"Se esperaban 4 preguntas en F{formulario}"
        )

        preguntas[formulario] = filas

    # Identificar columnas de estudiantes
    columnas = [
        c for c in range(7, base.shape[1])
        if pd.notna(base.iat[inicio, c])
        and str(base.iat[inicio, c]).strip()
    ]

    for col in columnas:

        original = str(base.iat[inicio, col]).strip()
        estudiante_id = f"EM_{len(participantes) + 1:03d}"

        tipo = (
            "id_sesion"
            if original.startswith("session_")
            else "nombre"
        )

        edad = pd.to_numeric(
            base.iat[etapas[1] + 1, col],
            errors="coerce"
        )

        genero = base.iat[etapas[1] + 2, col]

        participantes.append({
            "id_estudiante": estudiante_id,
            "tipo_origen": tipo,
            "bloque_origen": bloque,
            "edad": edad,
            "genero": genero
        })

        # Correspondencia privada para auditoría posterior
        mapa_privado.append({
            "id_estudiante": estudiante_id,
            "identificador_original": original
        })

        # Extraer las 12 respuestas posibles
        for formulario, filas in preguntas.items():
            for num_pregunta, fila in enumerate(filas, 1):

                valor = base.iat[fila, col]

                respuestas.append({
                    "id_estudiante": estudiante_id,
                    "formulario": formulario,
                    "pregunta": num_pregunta,
                    "variable": f"F{formulario}_P{num_pregunta}",
                    "respuesta_original": (
                        valor if pd.notna(valor) else pd.NA
                    ),
                    "fila_excel": fila + 1
                })

# Crear las tablas normalizadas
df_participantes = pd.DataFrame(participantes)
df_respuestas = pd.DataFrame(respuestas)
df_mapa_privado = pd.DataFrame(mapa_privado)

# Convertir las respuestas en columnas
tabla_respuestas = df_respuestas.pivot(
    index="id_estudiante",
    columns="variable",
    values="respuesta_original"
).reset_index()

tabla_respuestas.columns.name = None

# Merge relacional por identificador interno
dataset_base = df_participantes.merge(
    tabla_respuestas,
    on="id_estudiante",
    how="left",
    validate="one_to_one"
)

# Identificar las 12 variables de preguntas
columnas_preguntas = [
    f"F{f}_P{p}"
    for f in (1, 2, 3)
    for p in (1, 2, 3, 4)
]

# Auditar completitud
dataset_base["n_respuestas"] = (
    dataset_base[columnas_preguntas]
    .notna().sum(axis=1)
)

def clasificar_completitud(n):
    if n == 12:
        return "completo"
    if n == 0:
        return "sin_respuestas"
    return "parcial"

dataset_base["integridad"] = (
    dataset_base["n_respuestas"]
    .apply(clasificar_completitud)
)

# Comprobaciones
assert len(dataset_base) == 27
assert dataset_base["id_estudiante"].is_unique

print("Participantes:", len(dataset_base))
print("Respuestas presentes:",
      dataset_base["n_respuestas"].sum())

print("\nEstado de integridad:")
display(dataset_base["integridad"].value_counts())

display(dataset_base[
    ["id_estudiante", "tipo_origen",
     "edad", "genero",
     "n_respuestas", "integridad"]
])


Participantes: 27
Respuestas presentes: 196

Estado de integridad:


,count
integridad,
completo,13
parcial,9
sin_respuestas,5


,id_estudiante,tipo_origen,edad,genero,n_respuestas,integridad
0,EM_001,nombre,17.0,m,12,completo
1,EM_002,nombre,17.0,m,12,completo
2,EM_003,nombre,17.0,m,12,completo
3,EM_004,nombre,18.0,f,12,completo
4,EM_005,nombre,18.0,f,12,completo
5,EM_006,nombre,17.0,f,12,completo
6,EM_007,nombre,16.0,f,12,completo
7,EM_008,nombre,18.0,f,12,completo
8,EM_009,nombre,17.0,f,12,completo
9,EM_010,id_sesion,16.0,f,7,parcial


## Etapa 2: Limpieza y validación de datos

### Objetivo

Evaluar la calidad de los datos recopilados mediante EmotionLab, identificando valores faltantes, inconsistencias de formato y posibles anomalías que puedan afectar la construcción del dataset híbrido.

### Procedimiento

Se utiliza la biblioteca Pandas de Python para realizar las siguientes operaciones:

1. Verificación de identificadores únicos de estudiantes.
2. Identificación y cuantificación de respuestas faltantes.
3. Normalización de tipos de datos numéricos.
4. Estandarización de la representación del género.
5. Análisis de los valores mínimos, máximos y distribución de las respuestas.
6. Clasificación de registros según su nivel de completitud.
7. Generación de un reporte de calidad y trazabilidad.

### Criterios de tratamiento

Se conservan los 27 participantes originales, incluidos aquellos con respuestas parciales o faltantes.

Los valores ausentes se representan mediante `NaN` y no son reemplazados por valores estimados.

Las respuestas numéricas se conservan sin alterar su escala. No se aplican reglas de exclusión por rango hasta validar el significado y las opciones originales de cada pregunta.

Los datos originales permanecen disponibles para auditoría y comparación.

### Resultado esperado

Un dataset normalizado y auditado, acompañado de reportes de calidad, que servirá como entrada para la posterior homologación de variables.

#LIMPIEZA DE DATOS
Convierte los valores de las respuestas al tipo numérico apropiado y representa los faltantes como NaN, sin inventar respuestas ni modificar las escalas.

También verifica que ningún estudiante desaparezca durante la transformación.

In [ ]:
# Copia de trabajo: preservamos el dataset original
dataset_limpio = dataset_base.copy(deep=True)

# Las 12 variables del cuestionario
columnas_preguntas = [
    f"F{formulario}_P{pregunta}"
    for formulario in (1, 2, 3)
    for pregunta in (1, 2, 3, 4)
]

# 1. Verificar los identificadores
assert dataset_limpio["id_estudiante"].notna().all()
assert dataset_limpio["id_estudiante"].is_unique
assert len(dataset_limpio) == 27

# 2. Normalizar valores faltantes y tipos numéricos
errores_conversion = []

for columna in columnas_preguntas:

    originales = (
        dataset_limpio[columna]
        .astype("string")
        .str.strip()
    )

    # Unificar marcadores de ausencia
    faltantes = originales.str.lower().isin(
        ["", "nan", "none", "null", "n/a", "na", "-"]
    ).fillna(False)

    originales = originales.mask(faltantes)

    # Convertir números sin modificar sus valores
    convertidos = pd.to_numeric(
        originales, errors="coerce"
    )

    # Auditar valores que no pudieron convertirse
    error = originales.notna() & convertidos.isna()

    for id_estudiante in dataset_limpio.loc[
        error, "id_estudiante"
    ]:
        errores_conversion.append({
            "id_estudiante": id_estudiante,
            "variable": columna,
            "problema": "valor_no_numerico"
        })

    dataset_limpio[columna] = convertidos

# 3. Normalizar la edad
dataset_limpio["edad"] = pd.to_numeric(
    dataset_limpio["edad"], errors="coerce"
)

# 4. Estandarizar género conservando ausencias
dataset_limpio["genero"] = (
    dataset_limpio["genero"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# 5. Calcular nuevamente la completitud
dataset_limpio["n_respuestas"] = (
    dataset_limpio[columnas_preguntas]
    .notna()
    .sum(axis=1)
)

def evaluar_integridad(n):
    if n == 12:
        return "completo"
    elif n == 0:
        return "sin_respuestas"
    return "parcial"

dataset_limpio["integridad"] = (
    dataset_limpio["n_respuestas"]
    .apply(evaluar_integridad)
)

# 6. Verificar que no se perdieron participantes
assert len(dataset_limpio) == len(dataset_base)

print("Participantes:", len(dataset_limpio))
print("Respuestas presentes:",
      dataset_limpio["n_respuestas"].sum())
print("Errores de conversión:",
      len(errores_conversion))

print("\nEstado de integridad:")
display(dataset_limpio["integridad"].value_counts())

print("\nDistribución de género:")
display(dataset_limpio["genero"].value_counts(
    dropna=False
))


Participantes: 27
Respuestas presentes: 196
Errores de conversión: 0

Estado de integridad:


,count
integridad,
completo,13
parcial,9
sin_respuestas,5



Distribución de género:


,count
genero,
f,14
m,7
<NA>,6


# Generar una auditoría de calidad
Ahora vamos a crear dos reportes: uno por estudiante y otro por pregunta.

In [ ]:

# AUDITORÍA POR ESTUDIANTE

auditoria_estudiantes = dataset_limpio[
    ["id_estudiante", "tipo_origen",
     "edad", "genero", "n_respuestas", "integridad"]
].copy()

auditoria_estudiantes["respuestas_faltantes"] = (
    12 - auditoria_estudiantes["n_respuestas"]
)

auditoria_estudiantes["porcentaje_completitud"] = (
    auditoria_estudiantes["n_respuestas"] / 12 * 100
).round(2)

# AUDITORÍA POR VARIABLE

resumen_variables = []

for columna in columnas_preguntas:
    valores = dataset_limpio[columna]

    resumen_variables.append({
        "variable": columna,
        "respuestas_presentes": valores.notna().sum(),
        "respuestas_faltantes": valores.isna().sum(),
        "valor_minimo": valores.min(),
        "valor_maximo": valores.max(),
        "valores_distintos": valores.nunique()
    })

auditoria_variables = pd.DataFrame(resumen_variables)

# Valores altos que requieren verificar la escala
respuestas_largas = dataset_limpio.melt(
    id_vars=["id_estudiante"],
    value_vars=columnas_preguntas,
    var_name="variable",
    value_name="respuesta"
)

valores_a_revisar = respuestas_largas[
    respuestas_largas["respuesta"].ge(9).fillna(False)
].copy()

print("AUDITORÍA POR ESTUDIANTE")
display(auditoria_estudiantes)

print("\nAUDITORÍA POR VARIABLE")
display(auditoria_variables)

print("\nVALORES >= 9 (REVISIÓN DE ESCALA)")
display(valores_a_revisar)

print("\nErrores de conversión:")
display(pd.DataFrame(errores_conversion))


AUDITORÍA POR ESTUDIANTE


,id_estudiante,tipo_origen,edad,genero,n_respuestas,integridad,respuestas_faltantes,porcentaje_completitud
0,EM_001,nombre,17.0,m,12,completo,0,100.00
1,EM_002,nombre,17.0,m,12,completo,0,100.00
2,EM_003,nombre,17.0,m,12,completo,0,100.00
3,EM_004,nombre,18.0,f,12,completo,0,100.00
4,EM_005,nombre,18.0,f,12,completo,0,100.00
5,EM_006,nombre,17.0,f,12,completo,0,100.00
6,EM_007,nombre,16.0,f,12,completo,0,100.00
7,EM_008,nombre,18.0,f,12,completo,0,100.00
8,EM_009,nombre,17.0,f,12,completo,0,100.00
9,EM_010,id_sesion,16.0,f,7,parcial,5,58.33



AUDITORÍA POR VARIABLE


,variable,respuestas_presentes,respuestas_faltantes,valor_minimo,valor_maximo,valores_distintos
0,F1_P1,22,5,1,9,9
1,F1_P2,21,6,1,9,8
2,F1_P3,21,6,1,9,6
3,F1_P4,21,6,1,9,8
4,F2_P1,15,12,1,9,6
5,F2_P2,15,12,1,9,8
6,F2_P3,15,12,1,9,6
7,F2_P4,14,13,1,8,6
8,F3_P1,13,14,1,4,3
9,F3_P2,13,14,1,9,5



VALORES >= 9 (REVISIÓN DE ESCALA)


,id_estudiante,variable,respuesta
10,EM_011,F1_P1,9
14,EM_015,F1_P1,9
24,EM_025,F1_P1,9
37,EM_011,F1_P2,9
56,EM_003,F1_P3,9
90,EM_010,F1_P4,9
91,EM_011,F1_P4,9
95,EM_015,F1_P4,9
105,EM_025,F1_P4,9
118,EM_011,F2_P1,9



Errores de conversión:


""


## Etapa 3: Homologación de variables

### Objetivo

Estandarizar los nombres de las variables obtenidas de los cuestionarios de EmotionLab, facilitando su interpretación, integración y posterior utilización en modelos de Machine Learning.

### Procedimiento

Se establece una correspondencia entre los identificadores provisionales de las preguntas y sus nombres originales, utilizando como referencia los cuestionarios registrados por EmotionLab.

Las variables correspondientes a la evaluación inicial y de cierre se renombran según su significado.

Las preguntas cuyo significado original no está confirmado conservan sus identificadores provisionales y se registran como pendientes de validación.

### Criterios de homologación

- Equivalencia semántica entre las preguntas.
- Correspondencia con la etapa de evaluación.
- Conservación de los valores originales.
- Trazabilidad de las transformaciones.
- Identificación de variables pendientes de validación.

### Resultado esperado

Un dataset con nombres estandarizados y un diccionario de variables que documente las correspondencias realizadas.

La homologación no modifica los valores de las respuestas, no genera datos sintéticos ni realiza imputaciones.

In [ ]:

# =====================================================
# ETAPA 3: HOMOLOGACIÓN DE VARIABLES
# =====================================================

# 1. Definir las variables de los tres formularios

mapa_variables = {
    # Formulario 1: antes de presentar
    "F1_P1": "estado_pre",
    "F1_P2": "nerviosismo_pre",
    "F1_P3": "preocupacion_pre",
    "F1_P4": "preparacion_pre",

    # Formulario 2: después de dificultad fácil
    "F2_P1": "estado_facil",
    "F2_P2": "nerviosismo_facil",
    "F2_P3": "satisfaccion_facil",
    "F2_P4": "alivio_facil",

    # Formulario 3: después de dificultad difícil
    "F3_P1": "estado_dificil",
    "F3_P2": "nerviosismo_dificil",
    "F3_P3": "satisfaccion_dificil",
    "F3_P4": "alivio_dificil"
}

# 2. Definir las preguntas originales

textos_preguntas = [
    "¿Cómo te sientes hoy?",
    "¿Te sientes nervioso?",
    "¿Te sientes preocupado?",
    "¿Te sientes preparado?",

    "¿Cómo te sientes después de presentar?",
    "¿Te sientes nervioso?",
    "¿Te sientes satisfecho?",
    "¿Te sientes aliviado?",

    "¿Cómo te sientes después de presentar?",
    "¿Te sientes nervioso?",
    "¿Te sientes satisfecho?",
    "¿Te sientes aliviado?"
]

# 3. Definir las etapas de evaluación

etapas_formularios = (
    ["Antes de presentar"] * 4
    + ["Después de dificultad fácil"] * 4
    + ["Después de dificultad difícil"] * 4
)

# 4. Crear el diccionario de variables

diccionario_variables = pd.DataFrame({
    "variable_original": list(mapa_variables.keys()),
    "variable_estandarizada": list(mapa_variables.values()),
    "pregunta": textos_preguntas,
    "etapa": etapas_formularios,
    "escala_minima": 1,
    "escala_maxima": 9,
    "estado_validacion": "texto_confirmado"
})

# 5. Contabilizar respuestas disponibles

diccionario_variables["respuestas_disponibles"] = [
    int(dataset_limpio[col].notna().sum())
    for col in mapa_variables.keys()
]

# 6. Crear dataset homologado

dataset_etapa3 = dataset_limpio.rename(
    columns=mapa_variables
).copy()

# 7. Verificar conservación de respuestas

for original, nuevo in mapa_variables.items():
    assert dataset_limpio[original].equals(
        dataset_etapa3[nuevo]
    ), f"Error: se modificaron respuestas de {original}"

# 8. Auditar los valores de la escala

variables = list(mapa_variables.values())

revision_escala = dataset_etapa3.melt(
    id_vars="id_estudiante",
    value_vars=variables,
    var_name="variable",
    value_name="respuesta"
)

valores_invalidos = revision_escala[
    revision_escala["respuesta"].notna()
    & (
        ~revision_escala["respuesta"].between(1, 9)
        | (revision_escala["respuesta"] % 1 != 0)
    )
].copy()

# 9. Resumen de respuestas por formulario

variables_iniciales = variables[0:4]
variables_facil = variables[4:8]
variables_dificil = variables[8:12]

resumen_etapas = pd.DataFrame({
    "id_estudiante": dataset_etapa3["id_estudiante"],
    "respuestas_iniciales": (
        dataset_etapa3[variables_iniciales]
        .notna().sum(axis=1)
    ),
    "respuestas_facil": (
        dataset_etapa3[variables_facil]
        .notna().sum(axis=1)
    ),
    "respuestas_dificil": (
        dataset_etapa3[variables_dificil]
        .notna().sum(axis=1)
    )
})

resumen_etapas["total_respuestas"] = (
    resumen_etapas[
        [
            "respuestas_iniciales",
            "respuestas_facil",
            "respuestas_dificil"
        ]
    ].sum(axis=1)
)

# 10. Verificaciones finales

assert len(dataset_etapa3) == 27
assert dataset_etapa3["id_estudiante"].is_unique
assert len(diccionario_variables) == 12

assert (
    resumen_etapas["total_respuestas"].sum()
    == dataset_limpio["n_respuestas"].sum()
)

print("ETAPA 3 COMPLETADA")
print("Variables homologadas:", len(mapa_variables))
print("Participantes:", len(dataset_etapa3))
print("Respuestas fuera de escala:", len(valores_invalidos))

print("\nDiccionario de variables:")
display(diccionario_variables)

print("\nRespuestas por formulario:")
display(
    resumen_etapas[
        [
            "respuestas_iniciales",
            "respuestas_facil",
            "respuestas_dificil"
        ]
    ].sum()
)

print("\nValores que requieren revisión:")
display(valores_invalidos)


ETAPA 3 COMPLETADA
Variables homologadas: 12
Participantes: 27
Respuestas fuera de escala: 0

Diccionario de variables:


,variable_original,variable_estandarizada,pregunta,etapa,escala_minima,escala_maxima,estado_validacion,respuestas_disponibles
0,F1_P1,estado_pre,¿Cómo te sientes hoy?,Antes de presentar,1,9,texto_confirmado,22
1,F1_P2,nerviosismo_pre,¿Te sientes nervioso?,Antes de presentar,1,9,texto_confirmado,21
2,F1_P3,preocupacion_pre,¿Te sientes preocupado?,Antes de presentar,1,9,texto_confirmado,21
3,F1_P4,preparacion_pre,¿Te sientes preparado?,Antes de presentar,1,9,texto_confirmado,21
4,F2_P1,estado_facil,¿Cómo te sientes después de presentar?,Después de dificultad fácil,1,9,texto_confirmado,15
5,F2_P2,nerviosismo_facil,¿Te sientes nervioso?,Después de dificultad fácil,1,9,texto_confirmado,15
6,F2_P3,satisfaccion_facil,¿Te sientes satisfecho?,Después de dificultad fácil,1,9,texto_confirmado,15
7,F2_P4,alivio_facil,¿Te sientes aliviado?,Después de dificultad fácil,1,9,texto_confirmado,14
8,F3_P1,estado_dificil,¿Cómo te sientes después de presentar?,Después de dificultad difícil,1,9,texto_confirmado,13
9,F3_P2,nerviosismo_dificil,¿Te sientes nervioso?,Después de dificultad difícil,1,9,texto_confirmado,13



Respuestas por formulario:


,0
respuestas_iniciales,85
respuestas_facil,59
respuestas_dificil,52



Valores que requieren revisión:


,id_estudiante,variable,respuesta


Validación de la homolegación

## Etapa 4: Construcción del Dataset Híbrido

### Objetivo

Construir un dataset híbrido mediante la integración relacional de los registros reales obtenidos de EmotionLab, garantizando la correspondencia entre participantes, respuestas y variables homologadas.

### Metodología

Se aplica un procedimiento de integración relacional mediante la biblioteca Pandas de Python.

Se utilizan tres componentes:

1. **Tabla de participantes:** contiene los identificadores seudonimizados y las características generales de los estudiantes.
2. **Tabla de respuestas:** contiene los valores registrados para cada pregunta y participante.
3. **Diccionario de variables:** contiene los nombres estandarizados, las etapas de evaluación y el estado de validación de las preguntas.

Mediante operaciones `merge`, se relacionan las respuestas con sus correspondientes variables y participantes.

Posteriormente, se realiza una transformación de formato largo a formato ancho, generando una tabla consolidada donde cada fila corresponde a un estudiante.

### Consideraciones metodológicas

- Se conserva la totalidad de los 27 participantes identificados.
- No se generan registros ni respuestas sintéticas.
- Se preservan los valores faltantes.
- No se realizan imputaciones durante la construcción.
- No se relacionan estudiantes diferentes mediante coincidencias demográficas.
- Se mantiene la procedencia y trazabilidad de los registros.
- Las homologaciones preliminares permanecen sujetas a validación semántica.

### Resultado esperado

Una tabla consolidada de participantes y respuestas, acompañada de una representación relacional detallada, que permita efectuar las validaciones finales antes de su utilización en Machine Learning.

In [ ]:
# =====================================================
# ETAPA 4: CONSTRUCCIÓN DEL DATASET HÍBRIDO
# =====================================================

# 1. Recuperar las variables originales

columnas_preguntas = [
    f"F{f}_P{p}"
    for f in (1, 2, 3)
    for p in (1, 2, 3, 4)
]

# 2. Crear tabla de participantes

tabla_participantes = dataset_limpio[
    [
        "id_estudiante",
        "tipo_origen",
        "bloque_origen",
        "edad",
        "genero",
        "n_respuestas",
        "integridad"
    ]
].copy()

assert tabla_participantes["id_estudiante"].is_unique

# 3. Convertir respuestas a formato largo

tabla_respuestas = dataset_limpio.melt(
    id_vars=["id_estudiante"],
    value_vars=columnas_preguntas,
    var_name="variable_original",
    value_name="respuesta"
)

# 4. Recuperar diccionario de la Etapa 3

tabla_diccionario = diccionario_variables[
    [
        "variable_original",
        "variable_estandarizada",
        "pregunta",
        "etapa",
        "estado_validacion"
    ]
].copy()

assert tabla_diccionario["variable_original"].is_unique

# 5. Relacionar respuestas y variables

tabla_respuestas = tabla_respuestas.merge(
    tabla_diccionario,
    on="variable_original",
    how="left",
    validate="many_to_one"
)

assert tabla_respuestas[
    "variable_estandarizada"
].notna().all()

# 6. Relacionar respuestas con participantes

tabla_relacional = tabla_respuestas.merge(
    tabla_participantes[
        [
            "id_estudiante",
            "tipo_origen",
            "bloque_origen"
        ]
    ],
    on="id_estudiante",
    how="left",
    validate="many_to_one"
)

assert len(tabla_relacional) == 27 * 12

# 7. Transformar al formato ancho

respuestas_homologadas = tabla_relacional.pivot(
    index="id_estudiante",
    columns="variable_estandarizada",
    values="respuesta"
).reset_index()

respuestas_homologadas.columns.name = None

# 8. Crear el dataset real consolidado

dataset_hibrido = tabla_participantes.merge(
    respuestas_homologadas,
    on="id_estudiante",
    how="left",
    validate="one_to_one"
)

variables_finales = list(
    tabla_diccionario["variable_estandarizada"]
)

# 9. Validar la integración

assert len(dataset_hibrido) == 27
assert dataset_hibrido["id_estudiante"].is_unique

assert (
    dataset_hibrido[variables_finales]
    .notna().sum().sum()
    == dataset_limpio[columnas_preguntas]
    .notna().sum().sum()
)

# =====================================================
# GENERACIÓN SINTÉTICA EXPERIMENTAL
# =====================================================

# 10. Configuración reproducible

OBJETIVO = 600
SEMILLA_ALEATORIA = 42

rng = np.random.default_rng(SEMILLA_ALEATORIA)

# 11. Conservar los registros reales

reales = dataset_hibrido.copy()

reales["tipo_registro"] = "real"
reales["id_semilla"] = pd.NA

# 12. Seleccionar perfiles con respuestas

semillas = dataset_hibrido.loc[
    dataset_hibrido[variables_finales]
    .notna().any(axis=1)
].copy()

assert len(semillas) > 0

# 13. Calcular registros sintéticos necesarios

n_sinteticos = OBJETIVO - len(reales)

assert n_sinteticos >= 0

# 14. Seleccionar perfiles originales

indices = rng.integers(
    0,
    len(semillas),
    size=n_sinteticos
)

sinteticos = semillas.iloc[indices].copy()
sinteticos = sinteticos.reset_index(drop=True)

# 15. Conservar el identificador de origen

sinteticos["id_semilla"] = (
    sinteticos["id_estudiante"]
)

# 16. Asignar nuevos identificadores técnicos

sinteticos["id_estudiante"] = [
    f"SIN_{i:04d}"
    for i in range(1, n_sinteticos + 1)
]

sinteticos["tipo_registro"] = (
    "sintetico_exploratorio"
)

# 17. Modificar una respuesta por perfil sintético
# Utilizar únicamente valores observados de la misma
# pregunta. No se modifica la escala original.

for i in sinteticos.index:

    candidatas = [
        columna
        for columna in variables_finales

        if pd.notna(sinteticos.at[i, columna])
        and semillas[columna].dropna().nunique() > 1
    ]

    if not candidatas:
        continue

    columna = rng.choice(candidatas)

    valor_original = sinteticos.at[i, columna]

    alternativas = [
        valor
        for valor in semillas[columna].dropna().unique()
        if valor != valor_original
    ]

    if alternativas:
        sinteticos.at[i, columna] = rng.choice(
            alternativas
        )

# 18. Recalcular completitud

sinteticos["n_respuestas"] = (
    sinteticos[variables_finales]
    .notna().sum(axis=1)
)

def evaluar_integridad(n):
    if n == 12:
        return "completo"
    if n == 0:
        return "sin_respuestas"
    return "parcial"

sinteticos["integridad"] = (
    sinteticos["n_respuestas"]
    .apply(evaluar_integridad)
)

# 19. Construir dataset experimental

dataset_hibrido_600 = pd.concat(
    [reales, sinteticos],
    ignore_index=True
)

# 20. Verificar cantidades

assert len(dataset_hibrido_600) == 600
assert dataset_hibrido_600["id_estudiante"].is_unique

conteo = (
    dataset_hibrido_600["tipo_registro"]
    .value_counts()
)

assert conteo.get("real", 0) == 27
assert conteo.get("sintetico_exploratorio", 0) == 573

# 21. Generar resumen de integración

resumen_integracion = (
    dataset_hibrido_600.groupby("tipo_registro")
    .agg(
        registros=("id_estudiante", "count"),
        respuestas_presentes=("n_respuestas", "sum")
    )
    .reset_index()
)

print("ETAPA 4 COMPLETADA")
print("Total de registros:", len(dataset_hibrido_600))
print("Registros reales:", len(reales))
print("Registros sintéticos:", len(sinteticos))
print("Variables de cuestionario:", len(variables_finales))

print("\nResumen de integración:")
display(resumen_integracion)

print("\nVista del dataset híbrido:")
display(dataset_hibrido_600.head())


ETAPA 4 COMPLETADA
Total de registros: 600
Registros reales: 27
Registros sintéticos: 573
Variables de cuestionario: 12

Resumen de integración:


,tipo_registro,registros,respuestas_presentes
0,real,27,196
1,sintetico_exploratorio,573,5144



Vista del dataset híbrido:


,id_estudiante,tipo_origen,bloque_origen,edad,genero,n_respuestas,integridad,alivio_dificil,alivio_facil,estado_dificil,...,estado_pre,nerviosismo_dificil,nerviosismo_facil,nerviosismo_pre,preocupacion_pre,preparacion_pre,satisfaccion_dificil,satisfaccion_facil,tipo_registro,id_semilla
0,EM_001,nombre,1,17.0,m,12,completo,1,3,4,...,1,3,4,3,4,3,1,2,real,NaN
1,EM_002,nombre,1,17.0,m,12,completo,5,2,2,...,2,3,3,3,1,4,4,1,real,NaN
2,EM_003,nombre,1,17.0,m,12,completo,1,1,4,...,1,3,1,1,9,1,2,1,real,NaN
3,EM_004,nombre,1,18.0,f,12,completo,1,1,2,...,3,2,1,5,4,3,1,2,real,NaN
4,EM_005,nombre,1,18.0,f,12,completo,3,6,2,...,2,3,6,3,1,4,1,7,real,NaN


## Etapa 5: Validación final y exportación del dataset

### Objetivo

Verificar la integridad estructural y la trazabilidad del dataset híbrido experimental construido a partir de los datos reales de EmotionLab.

### Metodología

Se realizan comprobaciones automáticas sobre la cantidad de registros, unicidad de identificadores, procedencia de las observaciones, conservación de las respuestas originales y consistencia de los indicadores de completitud.

Se mantienen separadas la base real de 27 participantes y la base experimental ampliada a 600 registros.

### Consideraciones

Los 573 registros sintéticos fueron generados mediante remuestreo y modificación controlada de respuestas existentes. No constituyen observaciones independientes ni deben presentarse como evaluaciones reales de nuevos participantes.

La utilización de esta base para entrenamiento supervisado requerirá verificar la disponibilidad de etiquetas reales de intención de reutilización y aplicar procedimientos que eviten filtraciones de información.

### Entregables

- Dataset real seudonimizado.
- Dataset híbrido experimental.
- Diccionario de variables.
- Reporte de validación estructural.

In [ ]:
# =====================================================
# ETAPA 5: VALIDACIÓN Y EXPORTACIÓN DEL DATASET
# EMOTIONLAB - PROYECTO CAPSTONE
# =====================================================

# -----------------------------------------------------
# 1. VALIDACIÓN DE LA ESTRUCTURA
# -----------------------------------------------------

assert len(dataset_hibrido_600) == 600, (
    "El dataset debe contener 600 registros."
)

assert dataset_hibrido_600["id_estudiante"].is_unique, (
    "Existen identificadores duplicados."
)

assert len(variables_finales) == 12, (
    "Deben existir 12 variables de cuestionario."
)

assert len(diccionario_variables) == 12, (
    "El diccionario debe contener 12 preguntas."
)

assert all(
    columna in dataset_hibrido_600.columns
    for columna in variables_finales
), "Faltan variables en el dataset."

print("1. Estructura general validada.")


# -----------------------------------------------------
# 2. VALIDAR PROCEDENCIA
# -----------------------------------------------------

conteo = (
    dataset_hibrido_600["tipo_registro"]
    .value_counts()
)

assert conteo.get("real", 0) == 27, (
    "Deben existir 27 registros reales."
)

assert conteo.get("sintetico_exploratorio", 0) == 573, (
    "Deben existir 573 registros sintéticos."
)

print("2. Procedencia de registros validada.")


# -----------------------------------------------------
# 3. VALIDAR CANTIDAD DE RESPUESTAS
# -----------------------------------------------------

conteo_respuestas = (
    dataset_hibrido_600[variables_finales]
    .notna()
    .sum(axis=1)
)

assert (
    dataset_hibrido_600["n_respuestas"]
    .eq(conteo_respuestas)
).all(), (
    "Existen diferencias en el conteo de respuestas."
)

print("3. Cantidad de respuestas validada.")


# -----------------------------------------------------
# 4. VERIFICAR CONSERVACIÓN DE DATOS REALES
# -----------------------------------------------------

originales = (
    dataset_hibrido
    .set_index("id_estudiante")
    [variables_finales]
    .sort_index()
)

reales_integrados = (
    dataset_hibrido_600.loc[
        dataset_hibrido_600["tipo_registro"] == "real"
    ]
    .set_index("id_estudiante")
    [variables_finales]
    .sort_index()
)

pd.testing.assert_frame_equal(
    originales,
    reales_integrados,
    check_dtype=False
)

print("4. Respuestas reales conservadas correctamente.")


# -----------------------------------------------------
# 5. VALIDAR ESCALA DE RESPUESTAS
# -----------------------------------------------------

# Los tres formularios utilizan una escala de 1 a 9.
# Se permiten respuestas ausentes (NaN).
# Las respuestas presentes deben ser enteros entre 1 y 9.

respuestas_originales = (
    dataset_hibrido_600[variables_finales]
)

respuestas_numericas = respuestas_originales.apply(
    pd.to_numeric,
    errors="coerce"
)

# Detectar valores no numéricos
errores_conversion = (
    respuestas_originales.notna()
    & respuestas_numericas.isna()
)

# Detectar valores menores que 1 o mayores que 9
errores_rango = (
    respuestas_numericas.notna()
    & (
        (respuestas_numericas < 1)
        | (respuestas_numericas > 9)
    )
)

# Detectar números decimales
errores_decimales = (
    respuestas_numericas.notna()
    & (respuestas_numericas % 1 != 0)
)

# Unificar las verificaciones
mascara_invalidos = (
    errores_conversion
    | errores_rango
    | errores_decimales
)

total_invalidos = int(
    mascara_invalidos.sum().sum()
)

print("5. Respuestas inválidas detectadas:", total_invalidos)

# Mostrar valores problemáticos si existen
if total_invalidos > 0:

    print("\nValores que requieren revisión:")

    detalle_invalidos = (
        respuestas_originales
        .where(mascara_invalidos)
        .stack()
        .reset_index()
    )

    detalle_invalidos.columns = [
        "fila",
        "variable",
        "valor_invalido"
    ]

    display(detalle_invalidos)

assert total_invalidos == 0, (
    "Se encontraron respuestas fuera de escala "
    "o con formato incorrecto."
)

print("Escala de 1 a 9 validada correctamente.")


# -----------------------------------------------------
# 6. VALIDAR TRAZABILIDAD SINTÉTICA
# -----------------------------------------------------

registros_sinteticos = (
    dataset_hibrido_600.loc[
        dataset_hibrido_600["tipo_registro"]
        == "sintetico_exploratorio"
    ]
)

assert registros_sinteticos[
    "id_semilla"
].notna().all(), (
    "Hay registros sintéticos sin identificador de origen."
)

assert set(
    registros_sinteticos["id_semilla"]
).issubset(
    set(dataset_hibrido["id_estudiante"])
), (
    "Se detectaron identificadores de origen desconocidos."
)

print("6. Trazabilidad sintética validada.")


# -----------------------------------------------------
# 7. GENERAR RESUMEN FINAL
# -----------------------------------------------------

resumen_final = (
    dataset_hibrido_600.groupby("tipo_registro")
    .agg(
        registros=("id_estudiante", "count"),
        respuestas_presentes=("n_respuestas", "sum")
    )
    .reset_index()
)

total_respuestas = int(
    dataset_hibrido_600["n_respuestas"].sum()
)

print("7. Resumen de validación generado.")


# -----------------------------------------------------
# 8. VERIFICAR VARIABLE OBJETIVO
# -----------------------------------------------------

tiene_objetivo = (
    "intencion_reuso"
    in dataset_hibrido_600.columns
)


# -----------------------------------------------------
# 9. MOSTRAR RESULTADOS
# -----------------------------------------------------

print("\n======================================")
print("EMOTIONLAB - VALIDACIÓN FINAL")
print("======================================")

print("Registros totales:", len(dataset_hibrido_600))
print("Registros reales:", conteo.get("real", 0))

print(
    "Registros sintéticos:",
    conteo.get("sintetico_exploratorio", 0)
)

print("Variables del cuestionario:", len(variables_finales))
print("Respuestas presentes:", total_respuestas)
print("Respuestas inválidas:", total_invalidos)

print("\nResumen por procedencia:")
display(resumen_final)

print("\nDiccionario actualizado:")
display(diccionario_variables)

print(
    "\n¿Contiene variable intencion_reuso?",
    tiene_objetivo
)

if not tiene_objetivo:
    print(
        "AVISO: No existe una etiqueta confirmada "
        "de intención de reutilización."
    )

print("\nVALIDACIÓN ESTRUCTURAL COMPLETADA.")


# =====================================================
# 10. EXPORTACIÓN DE ENTREGABLES
# =====================================================

# Exportar los 27 registros reales
dataset_hibrido.to_csv(
    "EmotionLab_Dataset_Real.csv",
    index=False,
    encoding="utf-8-sig"
)

# Exportar los 600 registros experimentales
dataset_hibrido_600.to_csv(
    "EmotionLab_Dataset_Hibrido_600.csv",
    index=False,
    encoding="utf-8-sig"
)

# Exportar el diccionario de las 12 preguntas
diccionario_variables.to_csv(
    "EmotionLab_Diccionario.csv",
    index=False,
    encoding="utf-8-sig"
)

# Exportar el resumen de validación
resumen_final.to_csv(
    "EmotionLab_Validacion.csv",
    index=False,
    encoding="utf-8-sig"
)


# -----------------------------------------------------
# 11. COMPRIMIR LOS CUATRO ARCHIVOS
# -----------------------------------------------------

archivos = [
    "EmotionLab_Dataset_Real.csv",
    "EmotionLab_Dataset_Hibrido_600.csv",
    "EmotionLab_Diccionario.csv",
    "EmotionLab_Validacion.csv"
]

nombre_zip = (
    "EmotionLab_Dataset_Entregables_Actualizado.zip"
)

with zipfile.ZipFile(
    nombre_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for archivo in archivos:
        zipf.write(archivo)


# -----------------------------------------------------
# 12. DESCARGAR LOS ENTREGABLES
# -----------------------------------------------------

print("\n======================================")
print("EXPORTACIÓN COMPLETADA")
print("======================================")

print("Archivos incluidos:")

for archivo in archivos:
    print("-", archivo)

print("\nArchivo generado:", nombre_zip)

files.download(nombre_zip)

print("\nETAPA 5 FINALIZADA CORRECTAMENTE.")


1. Estructura general validada.
2. Procedencia de registros validada.
3. Cantidad de respuestas validada.
4. Respuestas reales conservadas correctamente.
5. Respuestas inválidas detectadas: 0
Escala de 1 a 9 validada correctamente.
6. Trazabilidad sintética validada.
7. Resumen de validación generado.

EMOTIONLAB - VALIDACIÓN FINAL
Registros totales: 600
Registros reales: 27
Registros sintéticos: 573
Variables del cuestionario: 12
Respuestas presentes: 5340
Respuestas inválidas: 0

Resumen por procedencia:


,tipo_registro,registros,respuestas_presentes
0,real,27,196
1,sintetico_exploratorio,573,5144



Diccionario actualizado:


,variable_original,variable_estandarizada,pregunta,etapa,escala_minima,escala_maxima,estado_validacion,respuestas_disponibles
0,F1_P1,estado_pre,¿Cómo te sientes hoy?,Antes de presentar,1,9,texto_confirmado,22
1,F1_P2,nerviosismo_pre,¿Te sientes nervioso?,Antes de presentar,1,9,texto_confirmado,21
2,F1_P3,preocupacion_pre,¿Te sientes preocupado?,Antes de presentar,1,9,texto_confirmado,21
3,F1_P4,preparacion_pre,¿Te sientes preparado?,Antes de presentar,1,9,texto_confirmado,21
4,F2_P1,estado_facil,¿Cómo te sientes después de presentar?,Después de dificultad fácil,1,9,texto_confirmado,15
5,F2_P2,nerviosismo_facil,¿Te sientes nervioso?,Después de dificultad fácil,1,9,texto_confirmado,15
6,F2_P3,satisfaccion_facil,¿Te sientes satisfecho?,Después de dificultad fácil,1,9,texto_confirmado,15
7,F2_P4,alivio_facil,¿Te sientes aliviado?,Después de dificultad fácil,1,9,texto_confirmado,14
8,F3_P1,estado_dificil,¿Cómo te sientes después de presentar?,Después de dificultad difícil,1,9,texto_confirmado,13
9,F3_P2,nerviosismo_dificil,¿Te sientes nervioso?,Después de dificultad difícil,1,9,texto_confirmado,13



¿Contiene variable intencion_reuso? False
AVISO: No existe una etiqueta confirmada de intención de reutilización.

VALIDACIÓN ESTRUCTURAL COMPLETADA.

EXPORTACIÓN COMPLETADA
Archivos incluidos:
- EmotionLab_Dataset_Real.csv
- EmotionLab_Dataset_Hibrido_600.csv
- EmotionLab_Diccionario.csv
- EmotionLab_Validacion.csv

Archivo generado: EmotionLab_Dataset_Entregables_Actualizado.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


ETAPA 5 FINALIZADA CORRECTAMENTE.
